# Day 2.5 — Citations and Abstention

Basic RAG can still answer from irrelevant evidence. We now require a structured answer that either cites retrieved chunks or explicitly abstains.

```text
Evidence sufficient → answer + citations
Evidence insufficient → abstain + no citations
```

## Before you begin

### Learning outcomes

Require attributable citations and treat insufficient evidence as a successful abstention.

Architecture reference: [D07](../../diagrams/source/day_02.md).

### Expected observation

Answerable input cites supplied sections; unanswerable input does not invent an answer.


## Concept briefing

## Citations and abstention

A citation should identify evidence the application actually supplied. Asking the model
to "always cite sources" is insufficient; the host should verify that returned citation
identifiers correspond to retrieved chunks. When evidence is missing, abstention is a
successful safety behavior. It tells downstream users that another information source or
human decision is required.


In [ ]:
import os,sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
here=Path.cwd().resolve(); candidates=[here,here/"day_02_knowledge_and_state",here.parent]
project_root=next(p for p in candidates if (p/"src"/"knowledge_agent").exists())
sys.path.insert(0,str(project_root/"src"))
from knowledge_agent.documents import load_markdown_corpus
from knowledge_agent.embeddings import SentenceTransformerEmbedder
from knowledge_agent.generation import OpenRouterGroundedGenerator
from knowledge_agent.retrieval import VectorIndex
chunks=load_markdown_corpus(project_root/"data"/"corpus")
index=VectorIndex(SentenceTransformerEmbedder(os.getenv("EMBEDDING_MODEL","sentence-transformers/all-MiniLM-L6-v2")))
index.add(chunks)
generator=OpenRouterGroundedGenerator()

## Answerable question

In [ ]:
question="How long are battery fault-event records retained?"
retrieved=index.search(question,3)
answer=generator.generate(question,retrieved)
print(answer.model_dump_json(indent=2))

## Unanswerable question

Nearest-neighbour search always returns something. The generator must decide whether that evidence actually supports an answer.

In [ ]:
unknown="What is the purchase price of the battery system?"
unknown_answer=generator.generate(unknown,index.search(unknown,3))
print(unknown_answer.model_dump_json(indent=2))

## Verify citations in application code

A model-generated citation is still data to validate. Check that each cited chunk was actually supplied.

In [ ]:
provided={item.chunk.chunk_id for item in retrieved}
cited={citation.chunk_id for citation in answer.citations}
print("citations supplied to model:", cited <= provided)
assert answer.abstained or cited <= provided

## Exercise and checkpoint

Test one supported and two unsupported questions. A supported answer must cite a supplied chunk; an abstention must contain no citations. Citations improve inspectability but do not prove the answer is correct—the next notebook measures behaviour on a golden set.

## Your turn

Add one unanswerable question and assert abstention plus absence of fabricated citations.

## Recap

Grounding needs application checks, not only an instruction to cite.
